# Notebook 1: XML Validation & Understanding

**Purpose:** Validate XML file before Auto Loader ingestion

**Checks:**
1. File exists and has valid size
2. XML can be read by Spark
3. rowTag is valid and records found
4. Schema inference and structure

**Output:** xml_path and row_tag for Notebook 2

## Configuration

In [ ]:
dbutils.widgets.text(
    "xml_path",
    "/Volumes/dev_automotive/landing/landing_raw/final_geografic.xml",
    "XML file path (volume)",
)
dbutils.widgets.text("row_tag", "record", "rowTag (e.g. record)")

xml_path = dbutils.widgets.get("xml_path").strip()
row_tag = dbutils.widgets.get("row_tag").strip()

print(f"xml_path: {xml_path}")
print(f"row_tag:  {row_tag}")
print()

## Step 1: File Exists, Readable, Size Check

In [ ]:
df_text = spark.read.text(xml_path)
line_count = df_text.count()

assert line_count > 0, "❌ XML file is empty or not readable"
print(f"✅ File readable. Lines: {line_count:,}")

try:
    file_info = dbutils.fs.ls(xml_path)[0]
    file_size_mb = round(file_info.size / (1024 * 1024), 2)
    print(f"✅ File size: {file_size_mb} MB")
    
    if file_size_mb > 1000:
        print(f"   ⚠️  Large file (>1GB)")
        
except Exception as e:
    print(f"⚠️  Size check failed: {e}")

print()

## Step 2: Schema Discovery (Spark XML)

In [ ]:
from pyspark.sql import functions as F

sample_df = (
    spark.read
    .format("xml")
    .option("rowTag", row_tag)
    .option("inferSchema", True)
    .load(xml_path)
)

cols = sample_df.columns
only_corrupt = cols == ["_corrupt_record"] or (len(cols) == 1 and "_corrupt_record" in cols)

if only_corrupt:
    print("⚠️  Malformed XML: Only _corrupt_record column inferred")
    print("    → Use Notebook 2 with 'Run healer = true'")
    print()
    sample_df.printSchema()
else:
    record_count = sample_df.count()
    assert record_count > 0, f"❌ rowTag '{row_tag}' not found in XML"
    
    print(f"✅ Records found: {record_count:,}")
    
    if "_corrupt_record" in cols:
        corrupt_count = sample_df.filter(F.col("_corrupt_record").isNotNull()).count()
        if corrupt_count > 0:
            corrupt_pct = round(corrupt_count/record_count*100, 1)
            print(f"   ⚠️  Corrupt records: {corrupt_count:,} ({corrupt_pct}%)")
    else:
        print(f"✅ No corrupt records detected")
    
    print()
    print("Schema:")
    sample_df.printSchema()
    
    print()
    print("Sample data (first 10 records):")
    sample_df.limit(10).show(truncate=False)

print()

## Summary

In [ ]:
print("="*70)
print("VALIDATION SUMMARY")
print("="*70)
print()

if not only_corrupt:
    print("✅ GO - XML is valid and ready for ingestion")
else:
    print("⚠️  GO WITH CAUTION - Malformed XML detected")
    print("    → Enable healer in Notebook 2")

print()
print("Next Steps:")
print("  1. Run Notebook 2 (Bronze Ingestion)")
print("  2. Use these parameters:")
print(f"     • xml_path: {xml_path}")
print(f"     • row_tag:  {row_tag}")

if only_corrupt:
    print(f"     • Run healer: true")

print()
print("="*70)

## Parameters for Notebook 2

In [ ]:
print("Parameters for Notebook 2:")
print(f"xml_path: {xml_path}")
print(f"row_tag:  {row_tag}")